In [1]:
import os
import pickle
import pandas as pd
import networkx as nx
import osmnx as ox
import h3
from shapely.geometry import mapping
from geopy.distance import geodesic

In [2]:
def polygon_to_h3(geom, resolution):
    """Polyfill a (Multi)Polygon into the set of H3 cell IDs at the given resolution."""
    geom_mapping = mapping(geom)
    if geom_mapping['type'] == 'Polygon':
        polygons = [geom_mapping]
    else:  # MultiPolygon
        polygons = [{'type': 'Polygon', 'coordinates': coords} for coords in geom_mapping['coordinates']]

    cells = set()
    for poly in polygons:
        cells |= h3.polyfill(poly, resolution, geo_json_conformant=True)
    return cells

# Get all the H3 IDs within Budapest at resolution 10
budapest_boundary = ox.geocode_to_gdf('Budapest, Hungary')
unique_h3_ids = polygon_to_h3(budapest_boundary.geometry.iloc[0], resolution=10)

# Coordinates of Deak Ferenc Square (lat, lon)
deak_ferenc_square = (47.4975, 19.0550)

In [3]:
#adding h3s present in opten that were not included in the boundary search 
opten_df = pd.read_pickle('../data/data_gen/opten_hex_level.pkl')
opten_h3_ids = set(opten_df['h3_10'].astype(str))

missing_from_polyfill = opten_h3_ids - set(unique_h3_ids)
unique_h3_ids = set(unique_h3_ids) | missing_from_polyfill

In [4]:
len(unique_h3_ids)

34516

In [7]:
# Load the street network for Budapest 
G_drive = ox.graph_from_place('Budapest, Hungary', network_type='drive')

In [8]:
# Calculate centrality measures for each graph.
betweenness_centrality_drive = nx.betweenness_centrality(G_drive, weight='length')
closeness_centrality_drive = nx.closeness_centrality(G_drive, distance='length')

In [9]:
# Find the Deak Ferenc Square node on network.
deak_node_drive = ox.distance.nearest_nodes(G_drive, deak_ferenc_square[1], deak_ferenc_square[0])

# compute the shortest-path distance to Deak Ferenc Square from every node 
# single-source Dijkstra from deak on the reversed graph 
dist_to_deak_drive = nx.single_source_dijkstra_path_length(G_drive.reverse(copy=True), deak_node_drive, weight='length')

In [ ]:
# Convert every H3 ID to coordinates
unique_h3_ids = list(unique_h3_ids)
h3_coords = [h3.h3_to_geo(h3_id) for h3_id in unique_h3_ids]
latitudes = [lat for lat, lon in h3_coords]
longitudes = [lon for lat, lon in h3_coords]

# Vectorized nearest-node lookups
h3_nodes_drive = ox.distance.nearest_nodes(G_drive, longitudes, latitudes)

distances_to_deak = [geodesic(deak_ferenc_square, coord).meters for coord in h3_coords]

city_char = pd.DataFrame({
    'h3_id': unique_h3_ids,
    'latitude': latitudes,
    'longitude': longitudes,
    'distance_to_deak': distances_to_deak,
    'h3_node_drive': h3_nodes_drive,
    'shortest_path_drive': [dist_to_deak_drive.get(n, float('inf')) for n in h3_nodes_drive],
    'betweenness_drive': [betweenness_centrality_drive.get(n) for n in h3_nodes_drive],
    'closeness_drive': [closeness_centrality_drive.get(n) for n in h3_nodes_drive]
})

In [ ]:
# Save one row per H3 level-10 hexagon with all city characteristics, so it can later be
# merged with the opten and smoothed mobility data
os.makedirs("../output/data", exist_ok=True)
city_char.to_csv("../output/data/city_char.csv", index=False)

In [ ]:
# Save the graphs to GraphML files
# to load: G = ox.load_graphml(filepath='../output/data/budapest_street_network_drive.graphml')
ox.save_graphml(G_drive, filepath='../output/data/budapest_street_network_drive.graphml')

# Save centrality dictionaries
with open('../output/data/betweenness_centrality_drive.pkl', 'wb') as pklfile:
    pickle.dump(betweenness_centrality_drive, pklfile)
with open('../output/data/closeness_centrality_drive.pkl', 'wb') as pklfile:
    pickle.dump(closeness_centrality_drive, pklfile)

In [13]:
city_char.head()

,h3_id,latitude,longitude,distance_to_deak,h3_node_drive,shortest_path_drive,betweenness_drive,closeness_drive
0,8a1e037b10affff,47.494820,18.959726,7184.951246,277762997,9448.748660,0.000315,0.000064
1,8a1e03602987fff,47.470279,19.221877,12935.946494,1838383915,14993.474731,0.000036,0.000066
2,8a1e0360c48ffff,47.473285,19.302738,18863.764826,1306637166,20788.450581,0.000000,0.000050
3,8a1e0362384ffff,47.525900,19.250484,15059.935155,305045185,17812.110179,0.000058,0.000057
4,8a1e0378c58ffff,47.473634,19.065404,2766.821905,36265378,3744.368142,0.001744,0.000101
